In [1]:
%pip install pytesseract


[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import PyPDF2

def analyze_pdf(pdf_path):
    """
    Basic PDF analysis to see what content we can extract.
    Prints raw content and attempts to identify potential data rows.
    """
    try:
        with open(pdf_path, 'rb') as file:
            # Create PDF reader
            reader = PyPDF2.PdfReader(file)
            
            # Go through each page
            for page_num in range(len(reader.pages)):
                print(f"\n--- Page {page_num + 1} ---")
                
                # Get the page
                page = reader.pages[page_num]
                
                # Extract text
                text = page.extract_text()
                
                # Print raw content
                print("\nRaw content:")
                print(text)
                
                # Print content line by line
                print("\nLine by line content:")
                for line in text.split('\n'):
                    if line.strip():  # Only print non-empty lines
                        print(f"LINE: {repr(line)}")  # repr() shows hidden characters
                
    except Exception as e:
        print(f"Error processing PDF: {e}")

# Example usage
if __name__ == "__main__":
    pdf_path = "/Users/rc/Desktop/pdf/pdf-examples/Drawing Material Takeoff Example.pdf"
    analyze_pdf(pdf_path)


--- Page 1 ---

Raw content:


Line by line content:

--- Page 2 ---

Raw content:
– 
” ”
”
”
” ” 
” 
® 
” 
” 
® 
” 
” 
® 
 
 
 
 
 
” 
®
“”

Line by line content:
LINE: '– '
LINE: '” ”'
LINE: '”'
LINE: '”'
LINE: '” ” '
LINE: '” '
LINE: '® '
LINE: '” '
LINE: '” '
LINE: '® '
LINE: '” '
LINE: '” '
LINE: '® '
LINE: '” '
LINE: '®'
LINE: '“”'

--- Page 3 ---

Raw content:


Line by line content:

--- Page 4 ---

Raw content:


Line by line content:

--- Page 5 ---

Raw content:


Line by line content:


In [11]:
%pip install PyPDF2 pytesseract Pillow PyMuPDF

  Using cached pymupdf-1.25.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (3.4 kB)
Using cached pymupdf-1.25.2-cp39-abi3-macosx_11_0_arm64.whl (18.6 MB)

[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
import PyPDF2
import pytesseract
from PIL import Image
import io
import fitz  # PyMuPDF
import re

def extract_text_from_pdf_images(pdf_path):
    """
    Extract text from images in a PDF using PyMuPDF for image extraction
    and Tesseract for OCR.
    """
    # Open the PDF with PyMuPDF
    doc = fitz.open(pdf_path)
    results = []
    
    for page_num in range(len(doc)):
        # Get the page
        page = doc[page_num]
        
        # Get images from the page
        image_list = page.get_images()
        
        # Process each image
        for img_index, img in enumerate(image_list):
            try:
                # Get the image reference
                xref = img[0]
                
                # Extract image
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]
                
                # Convert to PIL Image
                image = Image.open(io.BytesIO(image_bytes))
                
                # Perform OCR
                text = pytesseract.image_to_string(image)
                print(f"\nRaw OCR text from image {img_index + 1}, page {page_num + 1}:")
                print(text)  # This will help us see what's being extracted
                
                # Process each line of extracted text
                for line in text.split('\n'):
                    # Pattern to match number, quantity, unit, and description
                    pattern = r'(?P<no>\d+)?\s*(?P<qty>\d+(?:\.\d+)?)\s*(?P<unit>[A-Za-z]+)\s*(?P<description>.*)'
                    match = re.search(pattern, line)
                    
                    if match:
                        item = {
                            'no': match.group('no'),
                            'qty': match.group('qty'),
                            'unit': match.group('unit'),
                            'description': match.group('description').strip()
                        }
                        results.append(item)
                        
            except Exception as e:
                print(f"Error processing image {img_index + 1} on page {page_num + 1}: {e}")
                continue
    
    doc.close()
    return results

def display_results(results):
    """Display the extracted information in a formatted way."""
    print("\nExtracted Information:")
    print("-" * 50)
    for item in results:
        print(f"No: {item['no']}")
        print(f"Quantity: {item['qty']}")
        print(f"Unit: {item['unit']}")
        print(f"Description: {item['description']}")
        print("-" * 50)

def main():
    pdf_path = "/Users/rc/Desktop/pdf/pdf-examples/Drawing Material Takeoff Example.pdf"
    try:
        # Extract information
        results = extract_text_from_pdf_images(pdf_path)
        
        # Display results
        display_results(results)
        
    except Exception as e:
        print(f"Error processing PDF: {e}")

if __name__ == "__main__":
    main()


Extracted Information:
--------------------------------------------------


In [14]:
from pathlib import Path
from PIL import Image
import pytesseract

# Replace with your path
directory = '/Users/rc/Desktop/pdf/pdf-examples/Drawing Material Takeoff Example.pdf'

files = Path(directory).glob('*.png')
for file in files:
    print(file)
    print(pytesseract.image_to_string(Image.open(file)))
    print('\n----------\n')

In [18]:
# Extract text from images in the PDF
results = extract_text_from_pdf_images(pdf_path)

# Display the extracted text
display_results(results)


Extracted Information:
--------------------------------------------------
